# NYC Housing x Transportation: Data Story (Milestone 1)
**Course:** Data Engineering and Visualization, Winter 2026 | **Deadline:** 9 Nov 2026

**How to work in this notebook (read first)**
- Each member works ONLY inside their own section (marked with their name) to avoid edit clashes.
- Do not rename the shared variables: `df_housing_raw`, `df_transport_raw`, `df_housing`, `df_transport`, `merged`.
- Every decision (drop vs impute, outlier rule, join level) needs a markdown cell explaining **why**.
- Log every AI prompt you use in section 9.

| Member | Section |
|---|---|
| M1 Housing lead | 2, 3, 7 (RQ1-2) |
| M2 Transport lead | 2, 4, 7 (RQ3-4) |
| M3 Integration lead | 0, 1, 5, 6, 7 (RQ5-6) |
| M4 Website + story lead | 7 (RQ7-8), 8, 10 |

## 0. Setup and dataset links (M3)
- Housing: https://data.cityofnewyork.us/Housing-Development/DOB-NOW-Build-Job-Application-Filings/w9ak-ipjd
- Crashes: https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95

Run the next two cells first. The download takes a few minutes. It retries dropped connections and saves pages to `soda_cache/`, so if it fails just re-run the cell and it resumes. Everyone runs it once in their own Colab copy.

In [ ]:
import re, os, time, hashlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from urllib.parse import urlencode

# Both IDs verified on data.cityofnewyork.us (Oct 2026):
HOUSING_ID   = 'w9ak-ipjd'   # DOB NOW: Build - Job Application Filings (~969K rows, 95 columns)
TRANSPORT_ID = 'h9gi-nx95'   # Motor Vehicle Collisions - Crashes (~2.27M rows, 29 columns)
START_DATE   = '2022-01-01'  # sampling window start (justified in section 1)
API = 'https://data.cityofnewyork.us/resource/{}.csv'

def find_col(cols, candidates, regex=None, required=True):
    """Return the first matching column name. Column names differ between the CSV
    download and the API, so we look them up instead of hard-coding them."""
    cols = list(cols)
    for c in candidates:
        if c in cols:
            return c
    if regex:
        hits = [c for c in cols if re.search(regex, c, flags=re.I)]
        if hits:
            return hits[0]
    if required:
        raise KeyError(f'None of {candidates} (regex={regex}) found. Available columns: {cols}')
    return None

def soda_get(dataset_id, params, retries=6, wait=4):
    """Download one page. Retries on dropped connections ('Connection reset by peer', timeouts)."""
    url = API.format(dataset_id) + '?' + urlencode(params)
    for attempt in range(1, retries + 1):
        try:
            return pd.read_csv(url, low_memory=False)
        except Exception as e:
            if attempt == retries:
                raise
            print(f'    {type(e).__name__}: retry {attempt}/{retries - 1} in {wait * attempt}s')
            time.sleep(wait * attempt)

def probe_columns(dataset_id):
    """Download 5 rows just to learn the real column names."""
    return list(soda_get(dataset_id, {'$limit': 5}).columns)

CACHE_DIR = 'soda_cache'          # pages are saved here, so a re-run resumes instead of starting over
os.makedirs(CACHE_DIR, exist_ok=True)

def load_since(dataset_id, date_col, select=None, start=START_DATE, page=50_000, max_rows=3_000_000):
    """Download rows with date_col >= start in pages of 50k (stable order by :id).
    - select: list of columns to download. Slimming 95 columns to ~15 makes the download much faster and lighter.
    - each page is cached on disk; a dropped connection only repeats that one page."""
    pages, offset = [], 0
    while offset < max_rows:
        params = {'$where': f"{date_col} >= '{start}T00:00:00'", '$order': ':id', '$limit': page, '$offset': offset}
        if select:
            params['$select'] = ','.join(select)
        key = hashlib.md5(urlencode(params).encode()).hexdigest()[:12]
        path = os.path.join(CACHE_DIR, f'{dataset_id}_{key}.pkl')
        if os.path.exists(path):
            chunk = pd.read_pickle(path)
        else:
            chunk = soda_get(dataset_id, params)
            chunk.to_pickle(path)
        pages.append(chunk)
        print(f'  {dataset_id}: {offset + len(chunk):,} rows')
        if len(chunk) < page:
            break
        offset += page
    return pd.concat(pages, ignore_index=True)

In [ ]:
housing_cols   = probe_columns(HOUSING_ID)
transport_cols = probe_columns(TRANSPORT_ID)
print('HOUSING columns  :', housing_cols)       # <- everyone: read this once
print('TRANSPORT columns:', transport_cols)

# --- which columns to download (slim = faster, less memory, fewer connection drops) ---
HOUSING_DATE_COL = find_col(housing_cols, ['filing_date', 'filed_date'], regex=r'filing.*date|date.*filing')
HOUSING_EXTRA_COLS = []     # M1: add any other housing columns you want for EDA (names from the list printed above)
housing_core = ['job_filing_number', 'filing_status', 'house_no', 'street_name', 'borough', 'block', 'lot', 'bin', 'commmunity_board']
housing_pattern = r'^(zip|latitude|longitude|gis_lat|gis_lon|job_type|filing_date|current_status_date|initial_cost|proposed_dwelling|existing_dwelling|proposed_no_of_stories|building_type|work_type|council|census|nta)'
housing_select = [c for c in housing_cols if c in housing_core or c in HOUSING_EXTRA_COLS or re.search(housing_pattern, c, flags=re.I)]
for c in [HOUSING_DATE_COL, find_col(housing_cols, ['job_type'], regex=r'job.*type'),
          find_col(housing_cols, ['zip_code', 'zip', 'zipcode'], regex=r'^zip', required=False),
          find_col(housing_cols, ['latitude', 'gis_latitude'], regex=r'^(gis_)?lat', required=False),
          find_col(housing_cols, ['longitude', 'gis_longitude'], regex=r'^(gis_)?lon', required=False)]:
    if c and c not in housing_select:
        housing_select.append(c)

transport_wanted = ['collision_id', 'crash_date', 'crash_time', 'borough', 'zip_code', 'latitude', 'longitude', 'on_street_name',
                    'number_of_persons_injured', 'number_of_persons_killed', 'number_of_pedestrians_injured', 'number_of_pedestrians_killed',
                    'number_of_cyclist_injured', 'number_of_cyclist_killed', 'number_of_motorist_injured', 'number_of_motorist_killed']
transport_select = [c for c in transport_wanted if c in transport_cols]
print('\nDownloading', len(housing_select), 'housing columns:', housing_select)

df_housing_raw   = load_since(HOUSING_ID, HOUSING_DATE_COL, select=housing_select)
df_transport_raw = load_since(TRANSPORT_ID, 'crash_date', select=transport_select)
print(df_housing_raw.shape, df_transport_raw.shape)

## 1. Dataset overview and selection reasoning (M3 writes; M1 and M2 add details for their dataset)

| | Housing | Transportation |
|---|---|---|
| Dataset | DOB NOW: Build - Job Application Filings (`w9ak-ipjd`, catalog ID **H5**) | Motor Vehicle Collisions - Crashes (`h9gi-nx95`, catalog ID **T4**) |
| Size (NYC Open Data page, Oct 2026) | ~969K rows, 95 columns | ~2.27M rows, 29 columns |
| Geographic attributes | borough, street name, block, lot, BIN, community board (plus ZIP and latitude/longitude where present; the code detects them, and missing ZIPs are recovered from the nearest crash location) | borough, zip_code, latitude, longitude, on/off/cross street |
| Temporal attributes | `filing_date` | crash_date, crash_time |
| Both have >= 9 columns | yes | yes |

**Why this pair:** both are tied to the same places and months, so they can be compared at ZIP x month without matching individual rows. Question: *do areas with heavy construction activity also see more street-safety problems?*

**Sampling (required justification):** both tables are large, so we download rows from **2022-01-01** onward through the SoQL API in 100k-row pages. 2022 onward gives ~4 years (enough for seasonality and lags), keeps the data recent, and fits in Colab memory.

**Known issues to document**
- Crash table: ZIP and borough are often blank (recovered from coordinates where possible, see section 5), coordinates of 0,0 exist, and the table is preliminary data.
- **Crash feed:** Streetsblog reported in September 2026 that the crash dataset had stopped updating. In our download the crash data runs through September 2026 with no visible gap (about 6,400 to 7,300 crashes per month in 2026), so the feed appears to have resumed. Our pipeline still uses the common date window of both datasets and drops an unfinished last month as a safeguard.
- Housing table: column names differ slightly from the CSV download vs. the API (e.g. the community board column is spelled `commmunity_board`), so we look columns up with `find_col` instead of hard-coding them.

**Integration plan:** aggregate both to **ZIP x year-month**, then join on `zip` + `year_month` (details and alternatives in section 5).

## 2. Shared EDA helpers (M1 + M2)

In [ ]:
def overview(df, name):
    print(f'=== {name}: {df.shape[0]:,} rows x {df.shape[1]} cols ===')
    display(pd.DataFrame({'dtype': df.dtypes, 'nulls': df.isna().sum(),
                          'null_%': (df.isna().mean()*100).round(1), 'unique': df.nunique()}))

def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return q1 - k*i, q3 + k*i

### 2b. Shared cleaning helpers (M3 wrote these; **M1 and M2 should reuse them**)
`clean_zip`, `clean_borough`, `clean_coords`, `add_time_cols`, `fill_zip_nearest`. Using the same functions on both datasets guarantees the join keys match exactly (5-digit string ZIPs, upper-case borough names, NYC-box coordinates).

In [ ]:
from sklearn.neighbors import BallTree

BOROUGHS = ['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']
BORO_MAP = {'1': 'MANHATTAN', '2': 'BRONX', '3': 'BROOKLYN', '4': 'QUEENS', '5': 'STATEN ISLAND',
            'MN': 'MANHATTAN', 'BX': 'BRONX', 'BK': 'BROOKLYN', 'QN': 'QUEENS', 'SI': 'STATEN ISLAND'}
NYC_LAT, NYC_LON = (40.45, 40.95), (-74.30, -73.65)      # rough NYC bounding box (domain rule)

def clean_zip(s):
    """Keep the first 5 digits ('11201-1234' -> '11201'); anything outside NYC ZIP range -> NaN."""
    z = s.astype(str).str.extract(r'(\d{5})')[0]
    return z.where(pd.to_numeric(z, errors='coerce').between(10001, 11697))

def clean_borough(s):
    """Upper-case, strip, map numeric codes (1-5) and abbreviations; unknown -> NaN."""
    b = s.astype(str).str.strip().str.upper().str.replace(r'\.0$', '', regex=True).replace(BORO_MAP)
    return b.where(b.isin(BOROUGHS))

def clean_coords(lat, lon):
    """Coordinates outside the NYC box (including the 0,0 placeholder) are invalid -> NaN."""
    lat, lon = pd.to_numeric(lat, errors='coerce'), pd.to_numeric(lon, errors='coerce')
    ok = lat.between(*NYC_LAT) & lon.between(*NYC_LON)
    return lat.where(ok), lon.where(ok)

def add_time_cols(df, date_col):
    out = df.copy()
    out['date'] = pd.to_datetime(out[date_col], errors='coerce')
    out['year'], out['month'] = out['date'].dt.year, out['date'].dt.month
    out['year_month'] = out['date'].dt.strftime('%Y-%m')
    return out

def fill_zip_nearest(target, ref, max_km=0.5):
    """Recover a missing ZIP from the NEAREST reference point that has a ZIP (haversine BallTree).
    Only used if the nearest point is within max_km; otherwise the ZIP stays missing."""
    out = target.copy()
    out['zip'] = out['zip'].astype(object)                  # may be all-NaN float if the table had no ZIP column
    need = out['zip'].isna() & out['lat'].notna() & out['lon'].notna()
    r = ref.dropna(subset=['zip', 'lat', 'lon'])
    if need.sum() == 0 or len(r) == 0:
        return out
    tree = BallTree(np.radians(r[['lat', 'lon']].to_numpy()), metric='haversine')
    dist, idx = tree.query(np.radians(out.loc[need, ['lat', 'lon']].to_numpy()), k=1)
    km = dist[:, 0] * 6371.0088
    out.loc[need, 'zip'] = np.where(km <= max_km, r['zip'].to_numpy()[idx[:, 0]], None)
    return out

### Cleaning contract (read before cleaning)
Your cleaned table must end up in **`df_housing`** (M1) / **`df_transport`** (M2) with exactly these columns, so integration never breaks:

| Both | Housing only (M1) | Transport only (M2) |
|---|---|---|
| `date` (datetime), `year`, `month`, `year_month` ('2024-03'), `zip` (5-char string), `borough` (upper-case), `lat`, `lon` | `job_type` | `number_of_persons_injured`, `number_of_persons_killed`, `number_of_pedestrians_injured/killed`, `number_of_cyclist_injured/killed`, `number_of_motorist_injured/killed` (numeric, no nulls) |

`quick_clean_housing` / `quick_clean_transport` in section 5 show the minimum version. Do the real cleaning (with EDA, outlier handling and markdown justifications) and **overwrite** `df_housing` / `df_transport`. Do not change the column names.

## 3. HOUSING: EDA and pre-integration cleaning (**M1**)

In [ ]:
overview(df_housing_raw, 'Housing raw')
df_housing_raw.describe(include='all').T.head(30)

In [ ]:
# TODO (M1): EDA plots (job type counts, filings by borough, filings over time, missing-value chart)
# TODO (M1): cleaning, each step with a markdown cell explaining the decision:
#   - parse filing_date, create year / month / year_month
#   - standardise borough (upper-case, strip) and ZIP (5-digit string, drop invalid)
#   - drop exact duplicates (e.g. job_filing_number)
#   - missing values: drop vs impute (justify)
#   - outliers: IQR / domain rules (justify)
df_housing = df_housing_raw.copy()

## 4. TRANSPORT: EDA and pre-integration cleaning (**M2**)

In [ ]:
overview(df_transport_raw, 'Transport raw')
df_transport_raw.describe(include='all').T.head(30)

In [ ]:
# TODO (M2): EDA plots (crashes by borough/hour/weekday, injuries distribution, missing-value chart)
# TODO (M2): cleaning, each step with a markdown cell:
#   - parse crash_date, create year / month / year_month
#   - standardise borough and ZIP; many crashes have NO borough/ZIP -> decide: drop, or recover via lat/long (justify)
#   - lat/long = 0 or outside NYC bounding box -> invalid, treat as missing
#   - duplicates (collision_id), outliers in injury counts
df_transport = df_transport_raw.copy()

## 5. INTEGRATION (**M3**)
### 5a. Quick-clean stand-ins
These give a first working `merged_data.csv` today, so the website can be built on real data while M1 and M2 do their full cleaning.

In [ ]:
def quick_clean_transport(raw):
    """Minimal standardisation so integration can run NOW. M2 replaces this with the full cleaning
    (EDA, outliers, justified decisions) as long as the OUTPUT COLUMNS stay the same."""
    d = raw.copy()
    n0 = len(d)
    if 'collision_id' in d:
        d = d.drop_duplicates('collision_id')
    d = add_time_cols(d, 'crash_date')
    out = pd.DataFrame(index=d.index)
    out['date'], out['year'], out['month'], out['year_month'] = d['date'], d['year'], d['month'], d['year_month']
    out['zip'] = clean_zip(d['zip_code'])
    out['borough'] = clean_borough(d['borough'])
    out['lat'], out['lon'] = clean_coords(d['latitude'], d['longitude'])
    for c in ['number_of_persons_injured', 'number_of_persons_killed',
              'number_of_pedestrians_injured', 'number_of_pedestrians_killed',
              'number_of_cyclist_injured', 'number_of_cyclist_killed',
              'number_of_motorist_injured', 'number_of_motorist_killed']:
        out[c] = pd.to_numeric(d[c], errors='coerce').fillna(0) if c in d else 0
    missing_before = out['zip'].isna().sum()
    out = fill_zip_nearest(out, out)                       # recover ZIP from the nearest crash that has one
    print(f'[transport] rows {n0:,} -> {len(d):,} after dedupe | ZIP missing {missing_before:,} -> {out["zip"].isna().sum():,} after coordinate recovery')
    out = out.dropna(subset=['date', 'zip'])
    print(f'[transport] kept {len(out):,} rows with valid date + ZIP')
    return out

def quick_clean_housing(raw, zip_reference=None):
    """Minimal standardisation. M1 replaces it with the full cleaning (same output columns).
    zip_reference: cleaned transport table, used to recover ZIPs if the housing table lacks them."""
    cols = raw.columns
    date_col = find_col(cols, ['filing_date', 'filed_date'], regex=r'filing.*date|date.*filing')
    type_col = find_col(cols, ['job_type'], regex=r'job.*type')
    zip_col  = find_col(cols, ['zip_code', 'zip', 'zipcode'], regex=r'^zip', required=False)
    lat_col  = find_col(cols, ['latitude', 'gis_latitude'], regex=r'^(gis_)?lat', required=False)
    lon_col  = find_col(cols, ['longitude', 'gis_longitude'], regex=r'^(gis_)?lon', required=False)
    id_col   = find_col(cols, ['job_filing_number'], required=False)
    d = raw.copy()
    n0 = len(d)
    if id_col:
        d = d.drop_duplicates(id_col)
    d = add_time_cols(d, date_col)
    out = pd.DataFrame(index=d.index)
    out['date'], out['year'], out['month'], out['year_month'] = d['date'], d['year'], d['month'], d['year_month']
    out['job_type'] = d[type_col]
    out['zip'] = clean_zip(d[zip_col]) if zip_col else np.nan
    out['borough'] = clean_borough(d['borough']) if 'borough' in d else np.nan
    if lat_col and lon_col:
        out['lat'], out['lon'] = clean_coords(d[lat_col], d[lon_col])
    else:
        out['lat'], out['lon'] = np.nan, np.nan
    missing_before = out['zip'].isna().sum()
    if zip_reference is not None:
        out = fill_zip_nearest(out, zip_reference)
    print(f'[housing] rows {n0:,} -> {len(d):,} after dedupe | ZIP column found: {bool(zip_col)} | '
          f'ZIP missing {missing_before:,} -> {out["zip"].isna().sum():,} after recovery')
    out = out.dropna(subset=['date', 'zip'])
    print(f'[housing] kept {len(out):,} rows with valid date + ZIP')
    return out

In [ ]:
df_transport_q = quick_clean_transport(df_transport_raw)
df_housing_q   = quick_clean_housing(df_housing_raw, zip_reference=df_transport_q)

# Flip to True once M1 and M2 have delivered their full cleaning (same columns, see the contract above)
USE_FULL_CLEANING = False
housing_for_join   = df_housing   if USE_FULL_CLEANING else df_housing_q
transport_for_join = df_transport if USE_FULL_CLEANING else df_transport_q

### 5b. Aggregate and join at ZIP x month

In [ ]:
SCHEMA_COLS = ['zip', 'borough', 'year', 'month', 'year_month', 'lat', 'lon',
               'filings_new_building', 'filings_alteration', 'filings_other',
               'crashes_total', 'crashes_pedestrian', 'crashes_cyclist', 'crashes_motorist',
               'injuries', 'fatalities']

def housing_group(job_type):
    t = job_type.astype(str).str.strip().str.lower()
    is_nb  = t.str.contains('new building') | t.eq('nb')
    is_alt = t.str.contains('alteration') | t.str.fullmatch(r'a[123]')
    return np.select([is_nb, is_alt], ['new_building', 'alteration'], default='other')

def aggregate_housing(h):
    """One row per ZIP x month: number of job filings by group."""
    h = h.assign(job_group=housing_group(h['job_type']))
    g = h.groupby(['zip', 'year_month', 'job_group']).size().unstack('job_group', fill_value=0)
    for c in ['new_building', 'alteration', 'other']:
        if c not in g:
            g[c] = 0
    return g[['new_building', 'alteration', 'other']].add_prefix('filings_').reset_index()

def aggregate_transport(t):
    """One row per ZIP x month: crashes (all / with pedestrian / cyclist / motorist casualties), injuries, deaths."""
    t = t.assign(
        ped=(t['number_of_pedestrians_injured'] + t['number_of_pedestrians_killed']) > 0,
        cyc=(t['number_of_cyclist_injured'] + t['number_of_cyclist_killed']) > 0,
        mot=(t['number_of_motorist_injured'] + t['number_of_motorist_killed']) > 0)
    return (t.groupby(['zip', 'year_month'])
              .agg(crashes_total=('ped', 'size'), crashes_pedestrian=('ped', 'sum'),
                   crashes_cyclist=('cyc', 'sum'), crashes_motorist=('mot', 'sum'),
                   injuries=('number_of_persons_injured', 'sum'),
                   fatalities=('number_of_persons_killed', 'sum')).reset_index())

def build_zip_month_table(housing, transport):
    """Integrate at ZIP x month. Decisions (justify in the markdown cell below):
       1. keep only ZIPs present in BOTH datasets (a ZIP seen on one side only gives no comparison)
       2. use the common date window and drop the last month if it is incomplete
       3. build a COMPLETE ZIP x month grid so 'no filings / no crashes' is an explicit 0, not a missing row"""
    zips = sorted(set(housing['zip']) & set(transport['zip']))
    h, t = housing[housing['zip'].isin(zips)], transport[transport['zip'].isin(zips)]
    start = max(h['date'].min(), t['date'].min()).to_period('M')
    end_date = min(h['date'].max(), t['date'].max())
    end = end_date.to_period('M')
    if end_date.normalize() < end.end_time.normalize():     # last month not finished -> drop it
        end = end - 1
    months = [str(p) for p in pd.period_range(start, end, freq='M')]
    h, t = h[h['year_month'].isin(months)], t[t['year_month'].isin(months)]

    grid = pd.MultiIndex.from_product([zips, months], names=['zip', 'year_month']).to_frame(index=False)
    merged = (grid.merge(aggregate_housing(h), on=['zip', 'year_month'], how='left')
                  .merge(aggregate_transport(t), on=['zip', 'year_month'], how='left'))

    # ZIP-level attributes: borough = most common borough label, lat/lon = centroid of all point records
    pts = pd.concat([h[['zip', 'borough', 'lat', 'lon']], t[['zip', 'borough', 'lat', 'lon']]])
    zip_info = pts.groupby('zip').agg(
        borough=('borough', lambda s: s.mode().iloc[0] if s.notna().any() else np.nan),
        lat=('lat', 'mean'), lon=('lon', 'mean')).reset_index()
    merged = merged.merge(zip_info, on='zip', how='left')
    merged['year'] = merged['year_month'].str[:4].astype(int)
    merged['month'] = merged['year_month'].str[5:7].astype(int)

    info = dict(zips_housing=housing['zip'].nunique(), zips_transport=transport['zip'].nunique(),
                zips_kept=len(zips), window=f'{months[0]} to {months[-1]}', months=len(months),
                grid_rows=len(grid), housing_rows_used=len(h), transport_rows_used=len(t))
    return merged, info

In [ ]:
merged, info = build_zip_month_table(housing_for_join, transport_for_join)
print(info)
merged.head()

**Justification (M3: replace the numbers with `info` from your run)**
- **Join key `zip` + `year_month`.** Individual filings and crashes do not correspond to each other, so we compare *activity levels* per neighbourhood-month. ZIP is available on both sides (or recoverable from coordinates) and gives 217 ZIP codes, far more than the 5 boroughs; month keeps seasonality and allows lag analysis.
- **Alternatives considered:** borough x month (only 5 units, hides local patterns); row-level spatial join (a filing has no matching crash row, so it would be meaningless); census tract (too sparse per month).
- **Only ZIPs present in both datasets are kept** (a ZIP seen on one side gives no comparison).
- **Common window, partial last month dropped** (the crash feed ends earlier than the housing feed).
- **Complete ZIP x month grid:** months with no filings or no crashes are real zeros, so they are kept as explicit 0 rows instead of missing.
- **ZIP recovery:** missing crash ZIPs are filled from the nearest crash that has one (<= 0.5 km). If the housing table has no ZIP, it is filled from the nearest crash point: this borrows the crash table's ZIP boundaries, which we list as a limitation.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(f"""**Result of the integration:** {info['zips_kept']} ZIP codes appear in both datasets (housing had {info['zips_housing']} ZIPs, crashes had {info['zips_transport']}), \
over {info['months']} months ({info['window']}), giving a complete grid of {info['grid_rows']:,} ZIP-month rows. \
{info['housing_rows_used']:,} housing filings and {info['transport_rows_used']:,} crashes fall inside that grid."""))

## 6. POST-INTEGRATION CLEANING (**M3**) (required)

In [ ]:
COUNT_COLS = ['filings_new_building', 'filings_alteration', 'filings_other',
              'crashes_total', 'crashes_pedestrian', 'crashes_cyclist', 'crashes_motorist',
              'injuries', 'fatalities']

# The first 3 digits of a NYC ZIP identify the borough reliably, so we use them to catch wrong borough labels
ZIP_PREFIX_BOROUGH = {'100': 'MANHATTAN', '101': 'MANHATTAN', '102': 'MANHATTAN', '103': 'STATEN ISLAND',
                      '104': 'BRONX', '110': 'QUEENS', '111': 'QUEENS', '112': 'BROOKLYN',
                      '113': 'QUEENS', '114': 'QUEENS', '116': 'QUEENS'}

def fix_borough_by_zip(m):
    """Inconsistency check: where the borough label disagrees with the ZIP prefix, trust the ZIP."""
    m = m.copy()
    expected = m['zip'].str[:3].map(ZIP_PREFIX_BOROUGH)
    wrong = expected.notna() & (expected != m['borough'])
    if wrong.any():
        fixes = (m.loc[wrong, ['zip', 'borough']].drop_duplicates('zip').assign(corrected_to=expected[wrong]))
        print(f'Borough corrected from ZIP prefix for {fixes.shape[0]} ZIPs:'); print(fixes.to_string(index=False))
    m.loc[wrong, 'borough'] = expected[wrong]
    return m

def post_integration_clean(merged):
    """1) NaN from the left joins = no activity -> 0   2) fix dtypes   3) drop rows with no borough/coords
       4) remove duplicates   5) fix borough labels that contradict the ZIP   6) enforce the agreed schema/column order"""
    m = merged.copy()
    n0 = len(m)
    print('NaNs per column BEFORE:\n', m.isna().sum()[lambda s: s > 0].to_string())
    m[COUNT_COLS] = m[COUNT_COLS].fillna(0).astype(int)
    m['zip'] = m['zip'].astype(str).str.zfill(5)
    m = m.dropna(subset=['borough', 'lat', 'lon']).drop_duplicates(['zip', 'year_month'])
    m = fix_borough_by_zip(m)
    print(f'rows {n0:,} -> {len(m):,} (dropped rows without borough/coordinates, and duplicates)')
    return m[SCHEMA_COLS].sort_values(['zip', 'year_month']).reset_index(drop=True)

def validate(m):
    """Fail loudly if the table breaks the contract that the website and teammates rely on."""
    assert list(m.columns) == SCHEMA_COLS, 'columns differ from the schema contract'
    assert m.isna().sum().sum() == 0, 'nulls remain'
    assert not m.duplicated(['zip', 'year_month']).any(), 'duplicate ZIP-months'
    assert (m[COUNT_COLS] >= 0).all().all(), 'negative counts'
    assert (m['crashes_pedestrian'] <= m['crashes_total']).all(), 'pedestrian crashes exceed total'
    assert set(m['borough']) <= set(BOROUGHS), 'unexpected borough labels'
    print('VALID:', m.shape, '|', m['zip'].nunique(), 'ZIPs x', m['year_month'].nunique(), 'months')

def outlier_report(m, cols=('filings_new_building', 'crashes_total')):
    """IQR rule per ZIP total. Outliers are REPORTED, not removed: they are real high-activity ZIPs."""
    tot = m.groupby('zip')[list(cols)].sum()
    for c in cols:
        q1, q3 = tot[c].quantile([.25, .75]); hi = q3 + 1.5 * (q3 - q1)
        print(f'{c}: IQR upper bound {hi:,.0f} -> {(tot[c] > hi).sum()} ZIPs above it')

In [ ]:
final = post_integration_clean(merged)     # new NaNs from the joins -> 0, dtypes, redundant rows, schema order
validate(final)                            # fails loudly if the contract is broken
outlier_report(final)
final.describe().T

In [ ]:
# Sanity plots: do the integrated numbers look believable?
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
final.groupby('year_month')[['filings_new_building', 'crashes_total']].sum().plot(ax=ax[0], secondary_y='crashes_total', title='Monthly totals')
z = final.groupby('zip')[['filings_new_building', 'crashes_total']].sum()
sns.scatterplot(data=z, x='filings_new_building', y='crashes_total', ax=ax[1]); ax[1].set_title('ZIP totals')
sns.heatmap(final.pivot_table(index='borough', columns='month', values='crashes_total', aggfunc='sum'), ax=ax[2], cmap='YlOrRd')
ax[2].set_title('Crashes: borough x month')
plt.tight_layout(); plt.show()

In [ ]:
# Export: THIS file is what the website reads (put it in the repo at data/merged_data.csv)
final.to_csv('merged_data.csv', index=False)
try:
    from google.colab import files
    files.download('merged_data.csv')
except ImportError:
    pass
print(final.shape)

### 6b. Data health check (M3): numbers to quote in section 1 and the justification

In [ ]:
def health_report(final, housing_raw, transport_raw, housing_used, transport_used, info):
    """Numbers to quote in section 1 and the justification: row funnel, date windows, zeros, ZIP coverage."""
    print('--- ROW FUNNEL (how many records survived each step) ---')
    print(f"housing  : raw {len(housing_raw):,} -> cleaned {len(housing_used):,} -> inside window {info['housing_rows_used']:,}")
    print(f"transport: raw {len(transport_raw):,} -> cleaned {len(transport_used):,} -> inside window {info['transport_rows_used']:,}")
    print('\n--- DATE WINDOWS ---')
    print('housing  latest date :', housing_used['date'].max().date())
    print('crashes  latest date :', transport_used['date'].max().date(), '<- the crash feed may stop earlier than housing')
    print('integrated window    :', info['window'], f"({info['months']} months)")
    print('\n--- ZIP COVERAGE ---')
    print(f"ZIPs in housing {info['zips_housing']}, in crashes {info['zips_transport']}, kept (in both) {info['zips_kept']}")
    print(final.groupby('borough')['zip'].nunique().rename('ZIPs per borough').to_string())
    print('\n--- SHARE OF ZIP-MONTHS WITH ZERO ACTIVITY (quote this in your limitations) ---')
    for c in ['filings_new_building', 'filings_alteration', 'crashes_total', 'crashes_pedestrian', 'crashes_cyclist']:
        print(f'{c:24s} {(final[c] == 0).mean():6.1%}')
    print('\n--- TOTALS ---')
    print(final[['filings_new_building', 'filings_alteration', 'filings_other', 'crashes_total', 'injuries', 'fatalities']].sum().to_string())

health_report(final, df_housing_raw, df_transport_raw, housing_for_join, transport_for_join, info)

In [ ]:
from IPython.display import Markdown, display
# Every number below is computed from the data, so it stays correct if M1/M2 later change the cleaning.
n_t = df_transport_raw.drop_duplicates('collision_id').shape[0] if 'collision_id' in df_transport_raw else len(df_transport_raw)
n_h = df_housing_raw.drop_duplicates('job_filing_number').shape[0] if 'job_filing_number' in df_housing_raw else len(df_housing_raw)
unplaced_t = 1 - len(transport_for_join) / n_t
unplaced_h = 1 - len(housing_for_join) / n_h
yr = final.groupby('year')[['filings_new_building', 'filings_alteration']].sum()
full = yr.loc[2022:2025]
display(Markdown(f"""**Limitations of the integrated data (M3)**
- The crash feed was reported paused in 2026; the latest crash record in our download is from {transport_for_join['date'].max():%B %Y}, and the integrated window is {info['window']}.
- Filings are applications, not buildings finished or work started; timing can differ by months.
- {unplaced_t:.0%} of crash records and {unplaced_h:.0%} of housing filings could not be given a valid ZIP and date, and were dropped. Missing ZIPs were first recovered from the nearest record within 0.5 km.
- {(final['filings_new_building'] == 0).mean():.0%} of ZIP-months have zero new-building filings, so new-building results rest on few events in many ZIPs.
- Only ZIPs present in both datasets are kept ({info['zips_kept']} of {info['zips_housing']} housing ZIPs and {info['zips_transport']} crash ZIPs).
- New-building filings rose from {full.loc[2022, 'filings_new_building']:,} (2022) to {full.loc[2025, 'filings_new_building']:,} (2025) while alteration filings stayed between {full['filings_alteration'].min():,} and {full['filings_alteration'].max():,} a year. We do not read this as a construction boom: it may partly reflect changes in how filings are recorded in DOB NOW.
- Large ZIPs have more filings and more crashes; the lag analysis removes ZIP size, but the hotspot chart does not."""))

## 7. Research questions and visualizations (2 per member)
Write each question + 1 chart + 2-3 sentences of interpretation (no causal claims). The same function goes into your `charts_mX.py` file in the repo.

### M1: RQ1 and RQ2
*Question 1:* TODO

*Question 2:* TODO

In [ ]:
# TODO (M1): chart for RQ1

# TODO (M1): chart for RQ2

### M2: RQ3 and RQ4
*Question 1:* TODO

*Question 2:* TODO

In [ ]:
# TODO (M2): chart for RQ3

# TODO (M2): chart for RQ4

### M3: RQ5 and RQ6

**RQ5. Which ZIP codes rank in the top quarter for BOTH new-building filings and crashes?**
Why it matters: these are the places where new construction and street danger coincide, so they are where planners and safety teams should look first.
Method: total new-building filings and total crashes per ZIP over the study window, converted to percentile ranks among ZIPs; hotspot = top quarter on both.
*Findings:* 19 of 217 ZIPs (9%) rank in the top quarter on both new-building filings and crashes, compared with about 6% expected if the two were unrelated. Across all ZIPs the rank correlation is 0.54, a moderate positive relationship. The hotspots are in Brooklyn (10 ZIPs), Queens (7) and the Bronx (2), with none in Manhattan or Staten Island. The ZIPs with the most crashes among them are 11201, 11101, 11206, 11220 and 11377.
*Caution:* busier, larger ZIPs have more of everything, so this shows WHERE the two overlap, not that construction causes crashes.

**RQ6. Within a ZIP, do months with unusually many filings come before months with unusually many crashes?**
Why it matters: if construction raised crash risk, unusually busy months should be followed by unusually bad months.
Method: for each ZIP and month, subtract the ZIP's own average and the citywide average of that month (this removes ZIP size and seasonality), then correlate filings in month t with crashes in month t+lag, for lags 0 to 6.
*Findings:* After removing each ZIP's own average and each month's citywide average, the correlation between filings in one month and crashes in the following months is between 0.015 and 0.034 for every lag from 0 to 6 months (strongest at lag 3, r = 0.034 on 11,718 ZIP-month pairs; lag 0 is almost the same). Values this small mean unusually busy filing months were not followed by noticeably worse crash months.
*Caution:* correlation is not causation. Filings are paperwork, not construction started, and crash reporting depends on police recording.

**Taken together:** ZIPs with more new-building filings tend to also have more crashes (RQ5), but within a ZIP the timing shows almost no link (RQ6). This suggests the overlap reflects which neighbourhoods are dense and busy, not that construction timing drives crashes. We cannot establish causation with this data.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

import plotly.express as px
import plotly.graph_objects as go

HOT_Q = 0.75      # "high" = top quarter of ZIP codes
MAX_LAG = 6       # months


# ---------------------------------------------------------------- RQ1
def zip_summary(df):
    """One row per ZIP: totals, percentile rank in each domain, and a both-high flag."""
    d = df.groupby(["zip", "borough"], as_index=False)[["housing_metric", "transport_metric"]].sum()
    d["housing_pct"] = d["housing_metric"].rank(pct=True)
    d["transport_pct"] = d["transport_metric"].rank(pct=True)
    d["hotspot"] = (d["housing_pct"] >= HOT_Q) & (d["transport_pct"] >= HOT_Q)
    return d


def rq1(df):
    d = zip_summary(df)
    if d.empty:
        return go.Figure()
    d["marker_size"] = np.where(d["hotspot"], 15, 8)
    fig = px.scatter(d, x="housing_pct", y="transport_pct", color="borough", size="marker_size",
                     size_max=15, hover_name="zip",
                     hover_data={"housing_metric": True, "transport_metric": True,
                                 "housing_pct": ":.0%", "transport_pct": ":.0%", "marker_size": False},
                     labels={"housing_pct": "Construction activity (percentile among ZIPs)",
                             "transport_pct": "Crashes (percentile among ZIPs)",
                             "housing_metric": "Filings", "transport_metric": "Crashes"})
    # shade the "high on both" corner
    fig.add_shape(type="rect", x0=HOT_Q, x1=1.02, y0=HOT_Q, y1=1.02, fillcolor="rgba(200,68,47,0.10)",
                  line=dict(width=0), layer="below")
    n_hot, n = int(d["hotspot"].sum()), len(d)
    fig.add_annotation(x=1.0, y=1.0, xanchor="right", yanchor="bottom", showarrow=False,
                       text=f"{n_hot} of {n} ZIPs are high on both", font=dict(size=12, color="#c8442f"))
    fig.update_xaxes(tickformat=".0%", range=[0, 1.05])
    fig.update_yaxes(tickformat=".0%", range=[0, 1.05])
    return fig


# ---------------------------------------------------------------- RQ2
def _adjusted_panel(df, col):
    """ZIP x month matrix with each ZIP's own average and each month's citywide average removed.
    What is left = 'more or less than usual for this ZIP, in this month'."""
    wide = df.pivot_table(index="zip", columns="year_month", values=col, aggfunc="sum")
    months = pd.period_range(wide.columns.min(), wide.columns.max(), freq="M").astype(str)
    wide = wide.reindex(columns=months)               # missing months stay NaN, so lags never skip a gap
    w = wide.sub(wide.mean(axis=1), axis=0)
    w = w.sub(w.mean(axis=0), axis=1)
    return w.to_numpy()


def lag_correlations(df, max_lag=MAX_LAG):
    """Correlation between filings in month t and crashes in month t+lag, pooled over ZIPs."""
    if df.empty:
        return pd.DataFrame(columns=["lag", "r", "n"])
    h, t = _adjusted_panel(df, "housing_metric"), _adjusted_panel(df, "transport_metric")
    n_months, rows = h.shape[1], []
    for k in range(max_lag + 1):
        if n_months - k < 3:
            break
        a, b = h[:, : n_months - k], t[:, k:]
        ok = np.isfinite(a) & np.isfinite(b)
        r = np.nan
        if ok.sum() > 10:
            with np.errstate(invalid="ignore"):
                r = np.corrcoef(a[ok], b[ok])[0, 1]
        rows.append({"lag": k, "r": r, "n": int(ok.sum())})
    return pd.DataFrame(rows)


def rq2(df):
    lc = lag_correlations(df)
    if lc.empty or lc["r"].isna().all():
        return go.Figure().add_annotation(text="Not enough months for this filter", showarrow=False)
    fig = go.Figure(go.Bar(
        x=lc["lag"], y=lc["r"], text=lc["r"].round(3), textposition="outside",
        marker_color=np.where(lc["r"] >= 0, "#1f6f8b", "#c8442f"),
        customdata=lc["n"], hovertemplate="Lag %{x} months<br>r = %{y:.3f}<br>%{customdata:,} ZIP-month pairs<extra></extra>"))
    fig.update_layout(xaxis_title="Months between filings and crashes (0 = same month)",
                      yaxis_title="Correlation (ZIP- and month-adjusted)", xaxis=dict(dtick=1))
    fig.add_hline(y=0, line_width=1, line_color="#17212b")
    return fig

# Metric used in the notebook version of M3's questions: NEW-BUILDING filings vs ALL crashes
viz = final.assign(housing_metric=final['filings_new_building'], transport_metric=final['crashes_total'])

def summarize_rq1(d):
    hot = d[d['hotspot']].sort_values('transport_metric', ascending=False)
    n, n_hot = len(d), len(hot)
    rank_corr = d['housing_pct'].corr(d['transport_pct'])          # Pearson on ranks = Spearman
    print(f'{n_hot} of {n} ZIPs ({n_hot / n:.0%}) are in the top quarter on BOTH. If the two were unrelated we would expect ~6% (0.25 x 0.25).')
    print(f'Rank correlation between construction and crashes across ZIPs: {rank_corr:.2f}')
    print('Hotspots by borough:'); print(hot['borough'].value_counts().to_string())
    print('\nTop hotspot ZIPs:'); print(hot[['zip', 'borough', 'housing_metric', 'transport_metric']].head(10).to_string(index=False))

def summarize_rq2(lc):
    best = lc.loc[lc['r'].abs().idxmax()]
    print(lc.round(3).to_string(index=False))
    print(f"\nStrongest association: lag {int(best['lag'])} month(s), r = {best['r']:.3f} on {int(best['n']):,} ZIP-month pairs.")
    print('Rule of thumb: |r| < 0.1 is very weak. Report weak or zero results honestly; that is a finding too.')

# ---- RQ5
d_rq1 = zip_summary(viz)
rq1(viz).show()
summarize_rq1(d_rq1)

In [ ]:
# ---- RQ6
lc = lag_correlations(viz)
rq2(viz).show()
summarize_rq2(lc)

### M4: RQ7 and RQ8
*Question 1:* TODO

*Question 2:* TODO

In [ ]:
# TODO (M4): chart for RQ7

# TODO (M4): chart for RQ8

## 8. Story: question, discovery, limits, impact (M4 with everyone's input)
TODO

## 9. Prompt log (everyone)
| Member | Step | Prompt used |
|---|---|---|
| | | |
| M3 | Project planning | "Explain what should be done and distribute work among 4 members" |
| M3 | Starter code | "Build a Colab notebook skeleton and a basic Dash app that the team can edit" |
| M3 | Dataset check | "Open the links and confirm the dataset IDs, then help finish the Member 3 part" |
| M3 | Download fix | "Colab gives ConnectionResetError while downloading housing data, fix it" |
| M3 | Research questions | "Help finish Member 3: research questions and remaining tasks" |

## 10. Contributions
**Member 3 (integration and data layer):** designed the shared schema (`data_layer.py`) and the cleaning contract for M1/M2; wrote the robust downloader, ZIP-recovery helpers, ZIP x month aggregation, join, post-integration cleaning and validation; implemented the filter logic and the search-text parser for the website; built the merged dataset (`data/merged_data.csv`); research questions RQ5 (hotspot ZIPs in both domains) and RQ6 (lag relationship on ZIP- and month-adjusted data); reviewed teammates' cleaned tables for contract compliance and merged branches.

**Member 1:** TODO  |  **Member 2:** TODO  |  **Member 4:** TODO